#**📝 Notebook Overview**

---

This notebook demonstrates building a simple restaurant ordering system powered by AI. It covers:

- 📋 Viewing and managing the restaurant menu  
- 🛒 Adding/removing items from the cart with stock checks  
- 🧾 Generating unique order IDs and keeping order history  
- 🤖 Connecting AI tools for dynamic function calls  
- 💬 Handling conversational user queries with GPT-4o  
- ⚙️ Executing backend functions triggered by AI tool calls  
- 📊 Collecting and displaying results for analysis  
- ✅ Evaluating agent-generated responses using **LlumoClient** API to assess quality and accuracy
---
Use this notebook as a practical example to build interactive AI apps with real-time function execution and robust evaluation.


# **🔧 Import necessary libraries**


In [ ]:
# 🔧 Import necessary libraries

import json
import uuid
from openai import OpenAI


# **Tool Creation**
This module simulates a basic restaurant ordering system with functionalities to:

- 📋 View a menu of food and drink items  
- ➕ Add or remove items from the cart  
- 🧾 Place orders with unique IDs  
- 📦 View order history or clear the cart  

The system uses a Python dictionary to manage menu items and their stock, and stores all order details for future reference.

---




In [ ]:
import uuid  # For generating unique order IDs

# 🧾 Menu with item details: price, stock quantity, and description
menu = {
    "burger": {"price": 150, "stock": 10, "description": "Delicious beef burger"},
    "pizza": {"price": 300, "stock": 5, "description": "Cheesy pepperoni pizza"},
    "pasta": {"price": 250, "stock": 8, "description": "Creamy alfredo pasta"},
    "coke": {"price": 50, "stock": 20, "description": "Refreshing soft drink"},
    "sandwich": {"price": 120, "stock": 15, "description": "Grilled cheese sandwich"},
    "fries": {"price": 100, "stock": 12, "description": "Crispy golden french fries"},
    "mojito": {"price": 180, "stock": 10, "description": "Cool mint mojito"},
    "coffee": {"price": 120, "stock": 20, "description": "Hot brewed coffee"},
    "tea": {"price": 80, "stock": 25, "description": "Refreshing herbal tea"}
}

# 🛒 Global variables to manage cart and order history
cart = {}
orderHistory = {}

# 📋 Return the current menu
def getMenu():
    return str(menu)

# ➕ Add a specific quantity of an item to the cart, update stock
def addToCart(item, quantity):
    item = item.lower()
    if item in menu:
        if menu[item]["stock"] >= quantity:
            cart[item] = cart.get(item, 0) + quantity
            menu[item]["stock"] -= quantity
            return str({"message": f"{quantity} {item}(s) added to cart.", "cart": cart})
        else:
            return str({"error": f"Only {menu[item]['stock']} {item}(s) available."})
    return str({"error": "Item not available in menu."})

# ➖ Remove a specific quantity of an item from the cart, update stock
def removeFromCart(item, quantity):
    item = item.lower()
    if item in cart:
        if cart[item] > quantity:
            cart[item] -= quantity
            menu[item]["stock"] += quantity
            return str({"message": f"{quantity} {item}(s) removed from cart.", "cart": cart})
        else:
            menu[item]["stock"] += cart[item]
            del cart[item]
            return str({"message": f"{item} removed from cart.", "cart": cart})
    return str({"error": "Item not in cart."})

# ✅ Finalize the order and return a unique order ID
def getOrderDetails():
    if not cart:
        return str({"message": "Your cart is empty."})
    total = sum(menu[item]["price"] * qty for item, qty in cart.items())
    order_id = str(uuid.uuid4())[:8]
    orderHistory[order_id] = {"cart": cart.copy(), "total": total}
    cart.clear()
    return str({"orderId": order_id, "order": orderHistory[order_id]})

# 🧹 Clear the entire cart and restore item stock
def clearCart():
    for item, qty in cart.items():
        menu[item]["stock"] += qty
    cart.clear()
    return str({"message": "Cart has been cleared."})

# 📦 View previous orders (if any)
def viewOrderHistory():
    return str(orderHistory) if orderHistory else str({"message": "No past orders."})


# 🧠 Tool Definitions for Restaurant Order Assistant

This dictionary defines all available tools (functions) that can be called by an AI assistant. Each tool includes:

- 🔧 Function name
- 📝 Description
- 📦 Parameters with type and validation
- ✅ `strict=True` to enforce correct parameter input

These tools represent backend operations like getting the menu, adding/removing items to/from the cart, placing orders, clearing the cart, and viewing past orders.


In [ ]:
# Define tool specifications for AI-based function calling (OpenAI function-calling)

tools = [
    {"type": "function", "function": {"name": "getMenu", "description": "Get the restaurant menu.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "addToCart", "description": "Add an item to the cart.", "parameters": {"type": "object", "properties": {"item": {"type": "string"}, "quantity": {"type": "integer"}}, "required": ["item", "quantity"], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "removeFromCart", "description": "Remove an item from the cart.", "parameters": {"type": "object", "properties": {"item": {"type": "string"}, "quantity": {"type": "integer"}}, "required": ["item", "quantity"], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "getOrderDetails", "description": "Get the order details and generate an order ID.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "clearCart", "description": "Clear all items from the cart.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "viewOrderHistory", "description": "View past order history.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}, "strict": True}}
]

# **🔐 Initialize OpenAI Client**

Set up the OpenAI client using your API key. This allows secure access to OpenAI's language models for performing tasks such as response generation, tool calling, and more.

In [ ]:
# Set your OpenAI API key
key = "Your Api Key"  # 🔑 Replace with your actual API key

# Initialize the OpenAI client
client = OpenAI(api_key=key)


# 🤖 **Agent Simulation with Tool Calling using GPT-4o**

This section simulates a conversational interaction between a user and a GPT-4o agent that can call restaurant-related tools like `getMenu`, `addToCart`, `removeFromCart`, and `getOrderDetails`. The agent intelligently decides which tool to call based on the user query, executes the function, and provides a final response using the updated context.


In [ ]:
import json
import pandas as pd

# 🛠️ Function to execute a tool call based on the tool name and its arguments
def executeToolCall(toolCall):
    tool = toolCall.function                    # Extract the tool/function from the tool call
    args = json.loads(tool.arguments)           # Parse arguments from JSON string to dictionary

    # Call the matching local function based on tool name
    if tool.name == "getMenu":
        return getMenu()
    if tool.name == "addToCart":
        return addToCart(args["item"], args["quantity"])
    if tool.name == "removeFromCart":
        return removeFromCart(args["item"], args["quantity"])
    if tool.name == "getOrderDetails":
        return getOrderDetails()
    if tool.name == "clearCart":
        return clearCart()
    if tool.name == "viewOrderHistory":
        return viewOrderHistory()

    # Return error message if tool not recognized
    return "Unknown tool call."

# 📥 List of user queries simulating a conversation
user_queries = [
    "Show me the menu",
    "Add 2 burgers to my cart",
    "Add 1 coke to my cart",
    "Remove 1 burger from my cart",
    "Place my order"
]

results = []  # 📦 Store all query results

# 🔁 Loop through each user query and simulate interaction
for query in user_queries:
    messages = [{"role": "user", "content": query}]  # Start with user message

    # 💬 Call OpenAI's chat API with tool calling enabled
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools
    )

    aiMessage = response.choices[0].message
    messages.append(aiMessage)  # Append AI message

    output = ""

    # 🧠 If the AI calls a tool, execute it
    if aiMessage.tool_calls:
        for toolCall in aiMessage.tool_calls:
            toolResponse = executeToolCall(toolCall)
            messages.append({
                "role": "tool",
                "content": toolResponse,
                "tool_call_id": toolCall.id
            })

        # 🔁 Make a second model call after tool execution
        finalResponse = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools
        )
        output = finalResponse.choices[0].message.content
    else:
        output = aiMessage.content  # If no tool called, use direct response

    # 📊 Collect structured result data
    results.append({
        "user_query": query,
        "message_history": json.dumps(
            [m.model_dump() if hasattr(m, "model_dump") else {"role": m["role"], "content": m["content"]} for m in messages],
            indent=2
        ),
        "output": output,
        "all_tool_descriptions": json.dumps([t['function']['description'] for t in tools], indent=2)
    })

# 📄 Convert the results into a DataFrame for easy inspection
df = pd.DataFrame(results)

# 👀 Preview first few rows
print(df.head())


                     user_query  \
0              Show me the menu   
1      Add 2 burgers to my cart   
2         Add 1 coke to my cart   
3  Remove 1 burger from my cart   
4                Place my order   

                                     message_history  \
0  [\n  {\n    "role": "user",\n    "content": "S...   
1  [\n  {\n    "role": "user",\n    "content": "A...   
2  [\n  {\n    "role": "user",\n    "content": "A...   
3  [\n  {\n    "role": "user",\n    "content": "R...   
4  [\n  {\n    "role": "user",\n    "content": "P...   

                                              output  \
0  Here's the menu with descriptions and prices:\...   
1                 I've added 2 burgers to your cart.   
2  1 coke has been added to your cart. Your curre...   
3  1 burger has been removed from your cart. Now,...   
4  Your current order contains the following item...   

                               all_tool_descriptions  
0  [\n  "Get the restaurant menu.",\n  "Add an it...  
1  [\

In [ ]:
# raw dataframe
df


,user_query,message_history,output,all_tool_descriptions
0,Show me the menu,"[\n {\n ""role"": ""user"",\n ""content"": ""S...",Here's the menu with descriptions and prices:\...,"[\n ""Get the restaurant menu."",\n ""Add an it..."
1,Add 2 burgers to my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""A...",I've added 2 burgers to your cart.,"[\n ""Get the restaurant menu."",\n ""Add an it..."
2,Add 1 coke to my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""A...",1 coke has been added to your cart. Your curre...,"[\n ""Get the restaurant menu."",\n ""Add an it..."
3,Remove 1 burger from my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""R...","1 burger has been removed from your cart. Now,...","[\n ""Get the restaurant menu."",\n ""Add an it..."
4,Place my order,"[\n {\n ""role"": ""user"",\n ""content"": ""P...",Your current order contains the following item...,"[\n ""Get the restaurant menu."",\n ""Add an it..."


# **📊 Agent Response Evaluation using LlumoClient**

This section uses the `LlumoClient` to evaluate the AI agent's responses generated earlier. The evaluation takes place over the DataFrame `df`, which contains user queries, tool usage, and final outputs. A custom prompt template is provided to guide the evaluation.


In [ ]:
import pandas as pd

# Read the CSV file named 'openaiResults.csv' into a DataFrame
df = pd.read_csv("openaiResults.csv")

# Rename columns for better readability or consistency
df.rename(columns={
    "user_query": "query",
    "message_history": "messageHistory",
    "all_tool_descriptions": "tools"
}, inplace=True)

# Display the resulting DataFrame
print(df)


,query,messageHistory,output,tools
0,Show me the menu,"[\n {\n ""role"": ""user"",\n ""content"": ""S...",Here's the menu with descriptions and prices:\...,"[\n ""Get the restaurant menu."",\n ""Add an it..."
1,Add 2 burgers to my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""A...",I've added 2 burgers to your cart.,"[\n ""Get the restaurant menu."",\n ""Add an it..."
2,Add 1 coke to my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""A...",1 coke has been added to your cart. Your curre...,"[\n ""Get the restaurant menu."",\n ""Add an it..."
3,Remove 1 burger from my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""R...","1 burger has been removed from your cart. Now,...","[\n ""Get the restaurant menu."",\n ""Add an it..."
4,Place my order,"[\n {\n ""role"": ""user"",\n ""content"": ""P...",Your current order contains the following item...,"[\n ""Get the restaurant menu."",\n ""Add an it..."


### **📦 Install Llumo SDK**

Install the `llumo` Python package, which is required to evaluate AI agent responses using the Llumo platform.


In [ ]:
!pip install llumo

### **🚀 Evaluate Agent Responses Using LlumoClient**

In [ ]:
from llumo import LlumoClient

# Initialize the LlumoClient with your API key
client = LlumoClient(api_key="Your Api Key")

# Use the client to evaluate agent responses based on the provided DataFrame 'df'
# 'prompt_template' is used to format the prompt for each query dynamically
result = client.evaluateAgentResponses(
    dataframe=df,
    prompt_template="Provide answer for the given query: {{query}}"
)




======= Running evaluation for: Tool Reliability =======

======= Running evaluation for: Stepwise Progression =======

======= Running evaluation for: Tool Selection Accuracy =======

======= Running evaluation for: Final Task Alignment =======


### **📊 Displaying the evaluation results DataFrame**


In [ ]:
result

,query,messageHistory,output,tools,Tool Reliability,Tool Reliability Reason,Stepwise Progression,Stepwise Progression Reason,Tool Selection Accuracy,Tool Selection Accuracy Reason,Final Task Alignment,Final Task Alignment Reason
0,Show me the menu,"[\n {\n ""role"": ""user"",\n ""content"": ""S...",Here's the menu with descriptions and prices:\...,"[\n ""Get the restaurant menu."",\n ""Add an it...",100,The `getMenu` tool successfully executed and r...,100,The tool 'getMenu' is relevant to the user que...,100,The user requested the menu. The assistant use...,99,The assistant successfully retrieved and displ...
1,Add 2 burgers to my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""A...",I've added 2 burgers to your cart.,"[\n ""Get the restaurant menu."",\n ""Add an it...",100,The `addToCart` tool successfully added 2 burg...,99,The tool call `addToCart` is relevant to the u...,100,"The assistant used only the ""Add an item to th...",100,The assistant added 2 burgers to the cart as r...
2,Add 1 coke to my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""A...",1 coke has been added to your cart. Your curre...,"[\n ""Get the restaurant menu."",\n ""Add an it...",99,The `addToCart` tool successfully added the co...,100,The tool 'addToCart' is relevant to the user q...,100,"The assistant used only the ""Add an item to th...",100,The user requested to add a coke to their cart...
3,Remove 1 burger from my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""R...","1 burger has been removed from your cart. Now,...","[\n ""Get the restaurant menu."",\n ""Add an it...",100,The `removeFromCart` tool successfully removed...,99,The tool call `removeFromCart` directly addres...,99,The assistant used only the 'Remove an item fr...,100,The user requested removal of a burger from th...
4,Place my order,"[\n {\n ""role"": ""user"",\n ""content"": ""P...",Your current order contains the following item...,"[\n ""Get the restaurant menu."",\n ""Add an it...",100,The getOrderDetails tool successfully executed...,1,No tools were used to add items to the cart be...,2,"The assistant used the tool ""getOrderDetails"",...",1,The user requested to place an order. The assi...
